# Cleaning

In [1]:
import os
import pandas as pd
from openpyxl import load_workbook

INVOICES_DIR = 'invoices'

records = []

for filename in sorted(os.listdir(INVOICES_DIR)):
    if not filename.endswith('.xlsx'):
        continue

    filepath = os.path.join(INVOICES_DIR, filename)
    wb = load_workbook(filepath, data_only=True)
    ws = wb.active

    # Extract invoice-level fields from fixed cell coordinates
    invoice_no = ws['F4'].value
    inv_date   = ws['F5'].value
    due_date   = ws['F6'].value
    status     = ws['F7'].value
    client     = ws['C11'].value
    address    = ws['C12'].value
    contact    = ws['C13'].value

    # Extract line items — loop from row 16 until description is empty
    row = 16
    while True:
        desc = ws[f'B{row}'].value
        if not desc:
            break

        records.append({
            'invoice_no':  invoice_no,
            'date':        ws['F5'].value,
            'due_date':    due_date,
            'status':      status,
            'client':      client,
            'address':     address,
            'contact':     contact,
            'description': desc,
            'qty':         ws[f'C{row}'].value,
            'unit_price':  ws[f'D{row}'].value,
            'tax_rate':    ws[f'E{row}'].value,
        })
        row += 1

df = pd.DataFrame(records)

# Parse dates
df['date']     = pd.to_datetime(df['date'], dayfirst=True)
df['due_date'] = pd.to_datetime(df['due_date'], dayfirst=True)

# Compute line total in Python (formula values not recalculated by openpyxl)
df['line_total'] = df['qty'] * df['unit_price'] * (1 + df['tax_rate'])

df.to_csv('invoices_clean.csv', index=False)
print(df.head())

         invoice_no       date   due_date   status                     client  \
0  INV-20260101-001 2026-01-01 2026-01-31  PENDING  PT MITRA BISNIS Indonesia   
1  INV-20260101-001 2026-01-01 2026-01-31  PENDING  PT MITRA BISNIS Indonesia   
2  INV-20260102-002 2026-01-02 2026-02-01  PENDING  PT MITRA BISNIS Indonesia   
3  INV-20260102-002 2026-01-02 2026-02-01  PENDING  PT MITRA BISNIS Indonesia   
4  INV-20260102-002 2026-01-02 2026-02-01  PENDING  PT MITRA BISNIS Indonesia   

                                    address  \
0  Jl. Gatot Subroto No. 999, Bandung 40123   
1  Jl. Gatot Subroto No. 999, Bandung 40123   
2  Jl. Gatot Subroto No. 999, Bandung 40123   
3  Jl. Gatot Subroto No. 999, Bandung 40123   
4  Jl. Gatot Subroto No. 999, Bandung 40123   

                                  contact                description  qty  \
0  Budi Santoso | budi@mitraindonesia.com  Data Analysis & Reporting   15   
1  Budi Santoso | budi@mitraindonesia.com      UI/UX Design Services    9   